# Build FWD Dataset (DOCX + M4A)

This notebook block:
- reads transcript text from DOCX;
- converts M4A to WAV (16 kHz, mono);
- runs ASR with timestamps;
- aligns DOCX sentences to timestamped speech segments;
- exports segment WAV files to `data/fwd/fwd_audio`;
- builds a TSV manifest compatible with WhisperDataset (`path`, `transcription`).

In [6]:
%pip -q install python-docx pandas numpy soundfile librosa tqdm rapidfuzz faster-whisper


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
from __future__ import annotations

import re
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import soundfile as sf
from docx import Document
from faster_whisper import WhisperModel
from rapidfuzz import fuzz


PROJECT_ROOT = Path('/home/anna/python/MIPT/speach_recognition/FP')
FWD_DIR = PROJECT_ROOT / 'data' / 'fwd'
OUT_AUDIO_DIR = FWD_DIR / 'fwd_audio'
OUT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_TSV = FWD_DIR / 'fwd_manifest.tsv'
FULL_TSV = FWD_DIR / 'fwd_manifest_full.tsv'

PAIRS = [
    {
        'docx': FWD_DIR / 'SIM_Positives Denken  BG.docx',
        'audio': FWD_DIR / 'SIM_Positives Denken BG.m4a',
        'prefix': 'fwd_pos',
    },
    {
        'docx': FWD_DIR / 'SIM_Schweden und der Tabakkonsum_BG.docx',
        'audio': FWD_DIR / 'SIM_Schweden und der Tabakkonsum.m4a',
        'prefix': 'fwd_tabak',
    },
]


def read_docx_text(path: Path) -> str:
    doc = Document(path)
    chunks = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    text = '\n'.join(chunks)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def split_sentences(text: str) -> List[str]:
    # Sentence split that works reasonably for Bulgarian punctuation.
    parts = re.split(r'(?<=[.!?])\s+', text)
    parts = [p.strip() for p in parts if p.strip()]
    return parts


def ffmpeg_to_wav_16k_mono(inp: Path, out: Path) -> None:
    cmd = [
        'ffmpeg', '-y',
        '-i', str(inp),
        '-ac', '1',
        '-ar', '16000',
        '-vn',
        str(out),
    ]
    subprocess.run(cmd, check=True, capture_output=True)


def load_wav(path: Path) -> Tuple[np.ndarray, int]:
    audio, sr = sf.read(path)
    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)
    return audio.astype(np.float32), sr


@dataclass
class Seg:
    start: float
    end: float
    text: str


def transcribe_segments(model: WhisperModel, wav_path: Path, language: str = 'bg') -> List[Seg]:
    seg_iter, _ = model.transcribe(
        str(wav_path),
        language=language,
        beam_size=5,
        vad_filter=True,
        word_timestamps=True,
    )
    out: List[Seg] = []
    for s in seg_iter:
        # Prefer exact word boundaries when available to avoid cutting words.
        if s.words:
            starts = [w.start for w in s.words if w.start is not None]
            ends = [w.end for w in s.words if w.end is not None]
            if starts and ends:
                start = float(min(starts))
                end = float(max(ends))
            else:
                start = float(s.start)
                end = float(s.end)
        else:
            start = float(s.start)
            end = float(s.end)

        txt = (s.text or '').strip()
        if end > start and txt:
            out.append(Seg(start=start, end=end, text=txt))
    return out


def align_docx_to_segments(sentences: List[str], asr_segments: List[Seg], lookahead: int = 5) -> List[Seg]:
    # Monotonic fuzzy matching: for each audio segment choose the best next sentence.
    if not asr_segments:
        return []
    if not sentences:
        return [Seg(s.start, s.end, s.text) for s in asr_segments]

    aligned: List[Seg] = []
    i = 0
    for seg in asr_segments:
        if i >= len(sentences):
            break

        upper = min(len(sentences), i + lookahead)
        window = sentences[i:upper]

        best_local = 0
        best_score = -1
        for j, sent in enumerate(window):
            score = fuzz.token_set_ratio(seg.text.lower(), sent.lower())
            if score > best_score:
                best_score = score
                best_local = j

        chosen_idx = i + best_local
        aligned.append(Seg(start=seg.start, end=seg.end, text=sentences[chosen_idx]))
        i = chosen_idx + 1

    # If ASR produced fewer segments than sentences, append leftovers to the last one.
    if aligned and i < len(sentences):
        tail = ' '.join(sentences[i:]).strip()
        aligned[-1] = Seg(aligned[-1].start, aligned[-1].end, (aligned[-1].text + ' ' + tail).strip())

    return aligned


def export_segments(
    audio: np.ndarray,
    sr: int,
    segments: List[Seg],
    out_dir: Path,
    prefix: str,
    start_idx: int,
    min_sec: float = 1.2,
    max_sec: float = 20.0,
    pad_sec: float = 0.04,
) -> Tuple[List[Dict[str, Any]], int]:
    rows: List[Dict[str, Any]] = []
    idx = start_idx
    total_samples = len(audio)

    for seg in segments:
        s = max(0.0, seg.start - pad_sec)
        e = min(total_samples / sr, seg.end + pad_sec)
        dur = e - s

        text = re.sub(r'\s+', ' ', seg.text).strip()
        if dur < min_sec or dur > max_sec:
            continue
        if len(text) < 2:
            continue

        s_i = int(round(s * sr))
        e_i = int(round(e * sr))
        clip = audio[s_i:e_i]
        if clip.size == 0:
            continue

        out_name = f'{prefix}_{idx:06d}.wav'
        out_path = out_dir / out_name
        sf.write(str(out_path), clip, sr, subtype='PCM_16')

        rows.append({
            'path': str(out_path.resolve()),
            'transcription': text,
            'source': prefix,
            'lang': 'bg',
            'start_sec': round(s, 3),
            'end_sec': round(e, 3),
            'duration_sec': round(dur, 3),
        })
        idx += 1

    return rows, idx


def build_pair_dataset(
    model: WhisperModel,
    docx_path: Path,
    audio_path: Path,
    prefix: str,
    start_idx: int,
) -> Tuple[List[Dict[str, Any]], int]:
    wav_path = audio_path.with_suffix('.wav')
    ffmpeg_to_wav_16k_mono(audio_path, wav_path)

    ref_text = read_docx_text(docx_path)
    sentences = split_sentences(ref_text)

    asr_segments = transcribe_segments(model, wav_path, language='bg')
    aligned_segments = align_docx_to_segments(sentences, asr_segments, lookahead=6)

    audio, sr = load_wav(wav_path)
    if sr != 16000:
        raise ValueError(f'Expected 16000 Hz after conversion, got {sr}')

    rows, next_idx = export_segments(
        audio=audio,
        sr=sr,
        segments=aligned_segments,
        out_dir=OUT_AUDIO_DIR,
        prefix=prefix,
        start_idx=start_idx,
    )
    return rows, next_idx

In [8]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'

# Use a multilingual Whisper checkpoint with solid Bulgarian quality.
model = WhisperModel('large-v3', device=device, compute_type=compute_type)

all_rows: List[Dict[str, Any]] = []
next_index = 1

for pair in PAIRS:
    rows, next_index = build_pair_dataset(
        model=model,
        docx_path=pair['docx'],
        audio_path=pair['audio'],
        prefix=pair['prefix'],
        start_idx=next_index,
    )
    all_rows.extend(rows)
    print(f"{pair['prefix']}: {len(rows)} segments")

df_full = pd.DataFrame(all_rows)
if df_full.empty:
    raise RuntimeError('No segments were produced. Check source files and ffmpeg setup.')

# Full table for analysis/debug.
df_full.to_csv(FULL_TSV, sep='\t', index=False)

# Per-source manifests (path/transcription) for training by source.
manifest_paths: Dict[str, Path] = {}
for source_name, part in df_full.groupby('source'):
    part_manifest = FWD_DIR / f'{source_name}_manifest.tsv'
    part[['path', 'transcription']].to_csv(part_manifest, sep='\t', index=False)
    manifest_paths[source_name] = part_manifest

# Combined WhisperDataset-compatible manifest used by training code.
df_manifest = df_full[['path', 'transcription']].copy()
df_manifest.to_csv(MANIFEST_TSV, sep='\t', index=False)

print('Saved:')
print(f'  full:      {FULL_TSV}')
print(f'  combined:  {MANIFEST_TSV}')
for src, p in manifest_paths.items():
    print(f'  {src}: {p}')
print(f'  clips:     {OUT_AUDIO_DIR}')
print(f'  total:     {len(df_manifest)}')

fwd_pos: 15 segments
fwd_tabak: 12 segments
Saved:
  full:      /home/anna/python/MIPT/speach_recognition/FP/data/fwd/fwd_manifest_full.tsv
  combined:  /home/anna/python/MIPT/speach_recognition/FP/data/fwd/fwd_manifest.tsv
  fwd_pos: /home/anna/python/MIPT/speach_recognition/FP/data/fwd/fwd_pos_manifest.tsv
  fwd_tabak: /home/anna/python/MIPT/speach_recognition/FP/data/fwd/fwd_tabak_manifest.tsv
  clips:     /home/anna/python/MIPT/speach_recognition/FP/data/fwd/fwd_audio
  total:     27


# тут вообще все пошло не по плану алаймент модель не дала хороших результатов и пришлоась нарезать файлы вручную

In [ ]:
# Build text-only manifests for manual audio cutting (no auto alignment)
MANUAL_DIR = FWD_DIR / 'manual_manifests'
MANUAL_DIR.mkdir(parents=True, exist_ok=True)


def prepare_sentences_for_manual_cut(text: str, min_chars: int = 35) -> List[str]:
    sents = split_sentences(text)
    merged: List[str] = []
    buf = ''
    for s in sents:
        s = re.sub(r'\s+', ' ', s).strip()
        if not s:
            continue
        if not buf:
            buf = s
            continue
        # Merge tiny phrases with the next/previous sentence for easier manual cutting.
        if len(buf) < min_chars:
            buf = (buf + ' ' + s).strip()
        else:
            merged.append(buf)
            buf = s
    if buf:
        merged.append(buf)
    return merged


manual_frames = []
for pair in PAIRS:
    source = pair['prefix']
    txt = read_docx_text(pair['docx'])
    phrases = prepare_sentences_for_manual_cut(txt, min_chars=35)

    rows = []
    for i, phrase in enumerate(phrases, start=1):
        fname = f"{source}_manual_{i:04d}.wav"
        out_path = OUT_AUDIO_DIR / fname
        rows.append({
            'clip_id': f"{source}_{i:04d}",
            'path': str(out_path.resolve()),
            'transcription': phrase,
            'source': source,
            'lang': 'bg',
        })

    df_src = pd.DataFrame(rows)
    src_tsv = MANUAL_DIR / f"{source}_manual_manifest.tsv"
    df_src[['path', 'transcription']].to_csv(src_tsv, sep='\t', index=False)
    df_src.to_csv(MANUAL_DIR / f"{source}_manual_manifest_full.tsv", sep='\t', index=False)
    manual_frames.append(df_src)

    print(f"{source}: {len(df_src)} text segments")
    print(f"  -> {src_tsv}")

if manual_frames:
    df_manual_all = pd.concat(manual_frames, ignore_index=True)
    combined_tsv = MANUAL_DIR / 'fwd_manual_manifest.tsv'
    combined_full_tsv = MANUAL_DIR / 'fwd_manual_manifest_full.tsv'
    df_manual_all[['path', 'transcription']].to_csv(combined_tsv, sep='\t', index=False)
    df_manual_all.to_csv(combined_full_tsv, sep='\t', index=False)
    print(f"\nCombined -> {combined_tsv}")
    print(f"Combined full -> {combined_full_tsv}")

fwd_pos: 53 text segments
  -> /home/anna/python/MIPT/speach_recognition/FP/data/fwd/manual_manifests/fwd_pos_manual_manifest.tsv
fwd_tabak: 41 text segments
  -> /home/anna/python/MIPT/speach_recognition/FP/data/fwd/manual_manifests/fwd_tabak_manual_manifest.tsv

Combined -> /home/anna/python/MIPT/speach_recognition/FP/data/fwd/manual_manifests/fwd_manual_manifest.tsv
Combined full -> /home/anna/python/MIPT/speach_recognition/FP/data/fwd/manual_manifests/fwd_manual_manifest_full.tsv


In [12]:
# Combine manual manifests into one common manifest
manual_manifest_paths = [
    MANUAL_DIR / 'fwd_pos_manual_manifest.tsv',
    MANUAL_DIR / 'fwd_tabak_manual_manifest.tsv',
]

combined_frames = []
for manifest_path in manual_manifest_paths:
    df_part = pd.read_csv(manifest_path, sep='\t')
    df_part['source_manifest'] = manifest_path.name
    combined_frames.append(df_part)
    print(f'{manifest_path.name}: {len(df_part)} rows')

combined_manual_manifest = MANUAL_DIR / 'fwd_manual_manifest.tsv'
combined_manual_full_manifest = MANUAL_DIR / 'fwd_manual_manifest_full.tsv'

combined_df = pd.concat(combined_frames, ignore_index=True)
combined_df[['path', 'transcription']].to_csv(combined_manual_manifest, sep='\t', index=False)
combined_df.to_csv(combined_manual_full_manifest, sep='\t', index=False)

print('\nSaved:')
print(combined_manual_manifest)
print(combined_manual_full_manifest)
print(f'Total rows: {len(combined_df)}')

combined_df.head()

fwd_pos_manual_manifest.tsv: 53 rows
fwd_tabak_manual_manifest.tsv: 40 rows

Saved:
/home/anna/python/MIPT/speach_recognition/FP/data/fwd/manual_manifests/fwd_manual_manifest.tsv
/home/anna/python/MIPT/speach_recognition/FP/data/fwd/manual_manifests/fwd_manual_manifest_full.tsv
Total rows: 93


,path,transcription,source_manifest
0,/home/anna/python/MIPT/speach_recognition/FP/d...,"Скъпи колеги, През последните 20 години постоя...",fwd_pos_manual_manifest.tsv
1,/home/anna/python/MIPT/speach_recognition/FP/d...,"Учените вече са доказали, че позитивното мисле...",fwd_pos_manual_manifest.tsv
2,/home/anna/python/MIPT/speach_recognition/FP/d...,"Защото ние сме хора, а хората имат и негативни...",fwd_pos_manual_manifest.tsv
3,/home/anna/python/MIPT/speach_recognition/FP/d...,"Какво да правим тогава с негативните мисли, за...",fwd_pos_manual_manifest.tsv
4,/home/anna/python/MIPT/speach_recognition/FP/d...,На всички неведнъж са ни казвали: „Всичко ще б...,fwd_pos_manual_manifest.tsv


In [8]:
# Build manual-cut manifest for the human rights audiobook text.
from pathlib import Path
from typing import List
import re

import pandas as pd

AUDIOBOOKS_DIR = Path('/home/anna/python/MIPT/speach_recognition/FP/data/audiobooks')
AUDIOBOOKS_TEXT = AUDIOBOOKS_DIR / 'blg.txt'
AUDIOBOOKS_AUDIO_DIR = AUDIOBOOKS_DIR / 'prepared_audio'
AUDIOBOOKS_MANIFEST_TSV = AUDIOBOOKS_DIR / 'blg_manifest.tsv'
AUDIOBOOKS_FULL_TSV = AUDIOBOOKS_DIR / 'blg_manifest_full.tsv'

if not AUDIOBOOKS_AUDIO_DIR.exists():
    raise FileNotFoundError(f'Missing audio directory: {AUDIOBOOKS_AUDIO_DIR}')


def split_sentences(text: str) -> List[str]:
    parts = re.split(r'(?<=[.!?])\s+', text)
    parts = [part.strip() for part in parts if part.strip()]
    return parts


def read_plain_text(path: Path) -> str:
    text = path.read_text(encoding='utf-8')
    text = text.replace('\x0c', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def split_blg_sections(text: str) -> List[str]:
    marked = re.sub(r'\s*(Член\s+\d+)\s*', r'\n\1 ', text)
    parts = [part.strip() for part in marked.split('\n') if part.strip()]
    return parts


def prepare_blg_segments(text: str, min_chars: int = 80, max_chars: int = 220) -> List[str]:
    sections = split_blg_sections(text)
    segments: List[str] = []

    for section in sections:
        sentences = split_sentences(section)
        if not sentences:
            cleaned = re.sub(r'\s+', ' ', section).strip()
            if cleaned:
                segments.append(cleaned)
            continue

        buffer = ''
        for sentence in sentences:
            sentence = re.sub(r'\s+', ' ', sentence).strip()
            if not sentence:
                continue
            candidate = f'{buffer} {sentence}'.strip() if buffer else sentence
            if buffer and len(candidate) > max_chars:
                segments.append(buffer)
                buffer = sentence
            else:
                buffer = candidate
            if len(buffer) >= min_chars:
                segments.append(buffer)
                buffer = ''
        if buffer:
            if segments and len(buffer) < min_chars:
                segments[-1] = f"{segments[-1]} {buffer}".strip()
            else:
                segments.append(buffer)

    deduped = []
    for segment in segments:
        segment = re.sub(r'\s+', ' ', segment).strip()
        if segment:
            deduped.append(segment)
    return deduped


blg_text = read_plain_text(AUDIOBOOKS_TEXT)
blg_segments = prepare_blg_segments(blg_text)

blg_rows = []
for i, segment in enumerate(blg_segments, start=1):
    audio_path = AUDIOBOOKS_AUDIO_DIR / f'blg_{i:04d}.wav'
    blg_rows.append({
        'clip_id': f'blg_{i:04d}',
        'path': str(audio_path.resolve()),
        'transcription': segment,
        'source': 'blg',
        'lang': 'bg',
    })

df_blg = pd.DataFrame(blg_rows)
df_blg[['path', 'transcription']].to_csv(AUDIOBOOKS_MANIFEST_TSV, sep='\t', index=False)
df_blg.to_csv(AUDIOBOOKS_FULL_TSV, sep='\t', index=False)

print(f'\nHuman-rights manifest -> {AUDIOBOOKS_MANIFEST_TSV}')
print(f'Human-rights full manifest -> {AUDIOBOOKS_FULL_TSV}')
print(f'Prepared audio dir -> {AUDIOBOOKS_AUDIO_DIR}')
print(f'Total segments -> {len(df_blg)}')


Human-rights manifest -> /home/anna/python/MIPT/speach_recognition/FP/data/audiobooks/blg_manifest.tsv
Human-rights full manifest -> /home/anna/python/MIPT/speach_recognition/FP/data/audiobooks/blg_manifest_full.tsv
Prepared audio dir -> /home/anna/python/MIPT/speach_recognition/FP/data/audiobooks/prepared_audio
Total segments -> 60


In [10]:
# Build sentence-level manifest for the human rights audiobook text.
from pathlib import Path
from typing import List
import re

import pandas as pd

AUDIOBOOKS_DIR = Path('/home/anna/python/MIPT/speach_recognition/FP/data/audiobooks')
AUDIOBOOKS_TEXT = AUDIOBOOKS_DIR / 'blg.txt'
AUDIOBOOKS_AUDIO_DIR = AUDIOBOOKS_DIR / 'prepared_audio'
AUDIOBOOKS_MANIFEST_TSV = AUDIOBOOKS_DIR / 'blg_manifest.tsv'
AUDIOBOOKS_FULL_TSV = AUDIOBOOKS_DIR / 'blg_manifest_full.tsv'

if not AUDIOBOOKS_AUDIO_DIR.exists():
    raise FileNotFoundError(f'Missing audio directory: {AUDIOBOOKS_AUDIO_DIR}')


def split_sentences(text: str) -> List[str]:
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [part.strip() for part in parts if part.strip()]


def read_plain_text(path: Path) -> str:
    text = path.read_text(encoding='utf-8')
    text = text.replace('\x0c', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def split_blg_sections(text: str) -> List[str]:
    marked = re.sub(r'\s*(Член\s+\d+)\s*', r'\n\1 ', text)
    return [part.strip() for part in marked.split('\n') if part.strip()]


def split_long_clause_block(text: str) -> List[str]:
    text = re.sub(r'\s*(ВСEOБЩA\s+ДEКЛAРAЦИЯ\s+ЗA\s+ПРAВAТA\s+НA\s+ЧOВEКA)\s*', r'\n\1\n', text)
    text = re.sub(r'\s*(ПРEAМБЮЛ)\s*', r'\n\1 ', text)
    text = re.sub(r',\s+(?=Като взе предвид, че)', '\n', text)
    text = re.sub(r',\s+(?=OБЩOТO CЪБРAНИE)', '\n', text)
    return [part.strip(' ,') for part in text.split('\n') if part.strip(' ,')]


def prepare_blg_segments(text: str) -> List[str]:
    sections = split_blg_sections(text)
    segments: List[str] = []

    for section in sections:
        clause_parts = split_long_clause_block(section)
        for part in clause_parts:
            for sentence in split_sentences(part):
                sentence = re.sub(r'\s+', ' ', sentence).strip(' ,')
                if len(sentence) >= 2:
                    segments.append(sentence)

    return segments


blg_text = read_plain_text(AUDIOBOOKS_TEXT)
blg_segments = prepare_blg_segments(blg_text)

blg_rows = []
for i, segment in enumerate(blg_segments, start=1):
    audio_path = AUDIOBOOKS_AUDIO_DIR / f'blg_{i:04d}.wav'
    blg_rows.append({
        'clip_id': f'blg_{i:04d}',
        'path': str(audio_path.resolve()),
        'transcription': segment,
        'source': 'blg',
        'lang': 'bg',
    })

df_blg = pd.DataFrame(blg_rows)
df_blg[['path', 'transcription']].to_csv(AUDIOBOOKS_MANIFEST_TSV, sep='\t', index=False)
df_blg.to_csv(AUDIOBOOKS_FULL_TSV, sep='\t', index=False)

print(f'\nHuman-rights manifest -> {AUDIOBOOKS_MANIFEST_TSV}')
print(f'Human-rights full manifest -> {AUDIOBOOKS_FULL_TSV}')
print(f'Prepared audio dir -> {AUDIOBOOKS_AUDIO_DIR}')
print(f'Total segments -> {len(df_blg)}')
df_blg.head(12)


Human-rights manifest -> /home/anna/python/MIPT/speach_recognition/FP/data/audiobooks/blg_manifest.tsv
Human-rights full manifest -> /home/anna/python/MIPT/speach_recognition/FP/data/audiobooks/blg_manifest_full.tsv
Prepared audio dir -> /home/anna/python/MIPT/speach_recognition/FP/data/audiobooks/prepared_audio
Total segments -> 103


,clip_id,path,transcription,source,lang
0,blg_0001,/home/anna/python/MIPT/speach_recognition/FP/d...,ВСEOБЩA ДEКЛAРAЦИЯ ЗA ПРAВAТA НA ЧOВEКA,blg,bg
1,blg_0002,/home/anna/python/MIPT/speach_recognition/FP/d...,"ПРEAМБЮЛ Като взе предвид, че признаването на ...",blg,bg
2,blg_0003,/home/anna/python/MIPT/speach_recognition/FP/d...,"Като взе предвид, че пренебрегването и неуважа...",blg,bg
3,blg_0004,/home/anna/python/MIPT/speach_recognition/FP/d...,"Като взе предвид, че е необходимо правата на ч...",blg,bg
4,blg_0005,/home/anna/python/MIPT/speach_recognition/FP/d...,"Като взе предвид, че е необходимо да се нацърч...",blg,bg
5,blg_0006,/home/anna/python/MIPT/speach_recognition/FP/d...,"Като взе предвид, че народите на Oбединените н...",blg,bg
6,blg_0007,/home/anna/python/MIPT/speach_recognition/FP/d...,"Като взе предвид, че държавите-членки се задъл...",blg,bg
7,blg_0008,/home/anna/python/MIPT/speach_recognition/FP/d...,"Като взе предвид, че общото разбиране на тези ...",blg,bg
8,blg_0009,/home/anna/python/MIPT/speach_recognition/FP/d...,OБЩOТO CЪБРAНИE провъзгласява тази Всеобща дек...,blg,bg
9,blg_0010,/home/anna/python/MIPT/speach_recognition/FP/d...,Член 1 Bсички хора се раждат свободни и равни ...,blg,bg
